# 🔥 EnerGIS Scenario Studio

Interaktive Optimierung von industriellen Energiesystemen mit wissenschaftlichen Visualisierungen.

## 🎯 Quick Start

1. Alle Zellen mit **Run All** ausführen
2. Config-Pfade bei Bedarf anpassen
3. Ergebnisse werden automatisch in `saved_workflows/` gespeichert
4. Hochwertige Plots (PDF + SVG) für Publikationen werden erstellt

---

## 📦 Setup & Imports

In [ ]:
# Minimal-Bootstrap: Füge Projekt-Root zu sys.path hinzu
import sys
from pathlib import Path

# Finde Projekt-Root
current = Path.cwd()
for candidate in [current] + list(current.parents):
    if (candidate / 'energis').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

# Auto-Setup mit notebook_helpers
from energis.io.notebook_helpers import setup_notebook_environment

PROJECT_ROOT = setup_notebook_environment()
print("\n✅ Setup abgeschlossen")

In [ ]:
# Imports
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path

from energis.run import rolling_horizon as rh
from energis.config.merge import load_and_merge
from energis.io.notebook_helpers import (
    save_workflow_run,
    display_kpi_summary,
    create_and_display_dashboard
)

print("✅ Imports erfolgreich")
print(f"📊 Pandas Version: {pd.__version__}")

## ⚙️ Konfiguration

In [ ]:
# Config-Pfade
cfg_paths = [
    "configs/base.yaml",
    "configs/tech_catalog.yaml",
    "configs/sites/default.site.yaml",
    "configs/systems/baseline.system.yaml",
    "configs/scenarios/pf_then_rh.workflow.scenario.yaml",
]

# Optional: Overrides
overrides = None  # z.B.: {"run": {"solver": "glpk"}}

# Prüfen
print("📋 Konfigurationsdateien:")
all_exist = True
for p in cfg_paths:
    full_path = PROJECT_ROOT / p
    exists = full_path.exists()
    symbol = "✅" if exists else "❌"
    print(f"  {symbol} {p}")
    if not exists:
        all_exist = False

if not all_exist:
    raise FileNotFoundError("❌ Nicht alle Config-Dateien gefunden!")

print("\n✅ Konfiguration OK")

In [ ]:
# Config-Vorschau (spezifisch für Scenario Studio)
cfg_preview = load_and_merge(cfg_paths)

print("🔍 Config-Vorschau:")
print(f"  Solver:        {cfg_preview.get('run', {}).get('solver', 'N/A')}")
print(f"  Zeitschritt:   {cfg_preview.get('run', {}).get('dt_h', 'N/A')} h")
print(f"  CO2-Preis:     {cfg_preview.get('costs', {}).get('co2_price_eur_per_t', 'N/A')} EUR/t")
print(f"  Input-Datei:   {cfg_preview.get('site', {}).get('input_xlsx', 'N/A')}")
print(f"  Jahr:          {cfg_preview.get('site', {}).get('year_target', 'N/A')}")

# Systemkomponenten
sys_cfg = cfg_preview.get('system', {})
n_hp = len([hp for hp in sys_cfg.get('heat_pumps', []) if hp.get('enabled', True)])
n_gen = len([k for k,v in sys_cfg.get('generators', {}).items() if v.get('enabled', False)])
storage = sys_cfg.get('storage', {}).get('enabled', False)

print(f"\n🏭 Systemkomponenten:")
print(f"  Wärmepumpen:   {n_hp}")
print(f"  Generatoren:   {n_gen}")
print(f"  Speicher:      {'Ja' if storage else 'Nein'}")

## 🚀 Optimierung ausführen

In [ ]:
%%time
print("="*70)
print("▶ STARTE OPTIMIERUNG")
print("="*70)
print(f"⏰ Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

try:
    workflow = rh.run_workflow(cfg_paths, overrides=overrides)
    
    print("\n" + "="*70)
    print("✅ OPTIMIERUNG ERFOLGREICH ABGESCHLOSSEN")
    print("="*70)
    print(f"\n📊 Workflow: {' → '.join(workflow.plan.steps)}")
    
    optimization_successful = True
    
except Exception as e:
    print("\n" + "="*70)
    print("❌ FEHLER BEI DER OPTIMIERUNG")
    print("="*70)
    print(f"\n🔴 Fehler: {str(e)}\n")
    
    import traceback
    print("📋 Vollständiger Traceback:")
    traceback.print_exc()
    
    workflow = None
    optimization_successful = False
    
    print("\n💡 Troubleshooting:")
    print("  1. Prüfe ob Import_Data.xlsx existiert")
    print("  2. Prüfe Solver-Installation (gurobi/glpk)")
    print("  3. Prüfe ob alle Dependencies installiert sind")

## 💾 Workflow speichern & exportieren

In [ ]:
if optimization_successful and workflow:
    # Workflow-Name und Beschreibung
    WORKFLOW_NAME = "Scenario Studio Run"
    WORKFLOW_DESCRIPTION = "Interaktive Szenario-Analyse mit wissenschaftlichen Plots"
    
    # Workflow speichern (inkl. CSV, PDF, SVG Exports)
    workflow_dir = save_workflow_run(
        workflow,
        name=WORKFLOW_NAME,
        description=WORKFLOW_DESCRIPTION,
        config_paths=cfg_paths
    )
    
else:
    print("⚠️  Workflow-Speicherung übersprungen (Optimierung fehlgeschlagen)")

## 📈 Wissenschaftliche Visualisierungen

Erstelle hochwertige Plots in PDF + SVG Format für Publikationen.

In [ ]:
if optimization_successful and workflow:
    from energis.io.publication_plotter import export_publication_plots
    
    # Plot-Typen für wissenschaftliche Publikationen
    plot_types = [
        "input_data",           # Wärmebedarf, Strompreis, WRG Temp, WRG Q
        "results_combined",     # Bedarf + Erzeugung + Strompreis
        "heat_balance",         # Wärme-Bilanz
        "electric_balance",     # Elektrische Bilanz
        "storage",              # Speicher-Operation
    ]
    
    print("📊 Erstelle wissenschaftliche Plots (PDF + SVG)...\n")
    
    # Bestimme primäres Ergebnis
    primary_result = workflow.pf_result if workflow.pf_result else workflow.rh_result
    
    if primary_result:
        generated = export_publication_plots(
            outdir=str(workflow_dir),
            table=primary_result.table,
            series=primary_result.series,
            summary_sections=primary_result.summary if hasattr(primary_result, 'summary') else {},
            dpi=300,
            formats=("pdf", "svg"),  # Beide Vektorformate
            plot_types=plot_types
        )
        
        print(f"✅ {len(generated)} Plot-Typen erstellt:")
        for plot_type, files in generated.items():
            print(f"  • {plot_type}: {len(files)} Dateien")
            for f in files:
                print(f"    - {Path(f).name}")
        
        print(f"\n💡 Plots gespeichert in: {workflow_dir}")
        print("\n📚 Verwendung in Publikationen:")
        print("   • PDF: Direkt in LaTeX/Word/PowerPoint einfügen")
        print("   • SVG: Nachbearbeitung in Inkscape/Illustrator möglich")
        print("\n   LaTeX-Beispiel:")
        print("   \\includegraphics[width=\\textwidth]{heat_balance.pdf}")
    else:
        print("⚠️  Keine Ergebnisse für Plot-Erstellung verfügbar")
else:
    print("⚠️  Plot-Erstellung übersprungen (Optimierung fehlgeschlagen)")

## 📊 Key Performance Indicators

Detaillierte KPI-Analyse mit Kostenaufschlüsselung.

In [ ]:
if optimization_successful and workflow:
    # Nutze gemeinsame KPI-Funktion
    display_kpi_summary(workflow)
else:
    print("⚠️  Keine KPIs verfügbar")

## 🔍 Datenexploration (Optional)

Für detaillierte Datenanalyse und Korrelationen.

In [ ]:
if optimization_successful and workflow:
    # Erstelle DataFrame für Exploration
    primary_result = workflow.rh_result or workflow.mpc_result or workflow.pf_result
    
    if primary_result:
        ts = pd.DataFrame({
            'timestamp': primary_result.table.index,
            **{col: primary_result.table.data[col] for col in primary_result.table.columns},
            **primary_result.series,
        })
        ts.set_index('timestamp', inplace=True)
        
        print("📋 Zeitreihen-Daten (erste 10 Zeilen):\n")
        display(ts.head(10))
        
        print("\n📊 Statistische Zusammenfassung:\n")
        display(ts.describe())
        
        # DataFrame für weitere Analysen verfügbar machen
        print("\n💡 Tipp: DataFrame 'ts' ist jetzt für weitere Analysen verfügbar")
    else:
        print("⚠️  Keine Daten für Exploration verfügbar")
else:
    print("⚠️  Datenexploration übersprungen")

In [ ]:
# Korrelationsmatrix (optional)
if optimization_successful and workflow and 'ts' in locals():
    import matplotlib.pyplot as plt
    
    numeric_cols = ts.select_dtypes(include=[np.number]).columns
    
    if len(numeric_cols) > 1:
        fig, ax = plt.subplots(figsize=(12, 10))
        corr = ts[numeric_cols].corr()
        im = ax.imshow(corr, cmap='RdYlBu_r', aspect='auto', vmin=-1, vmax=1)
        
        ax.set_xticks(range(len(corr.columns)))
        ax.set_yticks(range(len(corr.columns)))
        ax.set_xticklabels(corr.columns, rotation=90, ha='right')
        ax.set_yticklabels(corr.columns)
        
        plt.colorbar(im, ax=ax)
        ax.set_title('🔗 Korrelationsmatrix', fontsize=16, fontweight='bold', pad=20)
        plt.tight_layout()
        plt.show()
else:
    print("⏭️  Korrelationsmatrix übersprungen")

## 🎛️ Interaktives Dashboard (Optional)

Zeigt das Dashboard mit allen interaktiven Visualisierungen.

In [ ]:
# Dashboard aktivieren/deaktivieren
SHOW_DASHBOARD = True

if optimization_successful and workflow and SHOW_DASHBOARD:
    try:
        dashboard = create_and_display_dashboard(
            workflow,
            title=f"Scenario Studio - {datetime.now().strftime('%Y-%m-%d')}"
        )
        
        print("\n💡 Dashboard-Features:")
        print("   • Wechsle zwischen Tabs für verschiedene Ansichten")
        print("   • Im Zeitreihen-Tab: Wähle Komponenten und Zeitbereich")
        print("   • Plots sind interaktiv: Zoom, Pan, Hover")
        print("   • Kosten-Tabelle ist sortierbar")
        
        # Dashboard anzeigen
        dashboard
        
    except ImportError as e:
        print(f"❌ Dashboard-Import-Fehler: {e}")
        print("   Installation: pip install panel holoviews bokeh plotly")
elif not SHOW_DASHBOARD:
    print("ℹ️  Dashboard deaktiviert (SHOW_DASHBOARD = False)")
else:
    print("⚠️  Dashboard kann nicht erstellt werden")

---

## 🎯 Nächste Schritte

### Sensitivitätsanalysen:
- CO2-Preis variieren
- Komponenten aktivieren/deaktivieren  
- Kapazitäten anpassen

### Weitere Analysen:
- Jahresdauerlinie erstellen
- Monats-Aggregation
- COP-Entwicklung analysieren

### Export & Sharing:
- Plots wurden als PDF+SVG für Publikationen exportiert
- Workflow wurde in `saved_workflows/` gespeichert
- Dashboard kann als Webapp gestartet werden: `panel serve scenario_studio.ipynb --show`

---